# Week 3 · Notebook 1 — Financial Data & Feature Engineering
### Multi-Agent Forecasting Project

**Name:** Mayank Raj  
**Date:** 2026-06-24

---
From this week the project becomes **real**: we forecast the daily moves of the **S&P 500** (via the `SPY` ETF) using live market data.

This is the first of six notebooks. Each one builds a single piece of the final multi-agent system. By the end of all six, the notebooks *together* are the project. This notebook builds the **data + feature layer** that every later agent will consume.

**Topics covered:**
1. From prices to **log returns** — and why we never forecast raw prices
2. Volatility and the **VIX** fear index
3. The **RSI** momentum indicator (coded from scratch)
4. The **MACD** trend indicator
5. **Bollinger Band width** as a volatility feature
6. Assembling a clean **feature matrix** — with zero future leakage

> **The golden rule of this project:** every feature you build to predict day *t* may only use information available *before* day *t*. One stray un-shifted column and your backtest becomes a fantasy. We will check for this explicitly.

In [ ]:
# Run this cell first — installs and imports everything you need.
!pip -q install yfinance

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

START = '2010-01-01'
END   = '2024-12-31'
print('Setup complete.')

---
## Section 1 — From Prices to Log Returns

Stock **prices** are non-stationary: their mean and variance drift over time (SPY was ~$100 in 2010 and ~$580 in 2024). Models trained on one price range generalise terribly to another.

**Returns** fix this. The *log return* on day *t* is:

$$ r_t = \log\left( \frac{P_t}{P_{t-1}} \right) = \log P_t - \log P_{t-1} $$

Log returns are (approximately) stationary, they are additive over time, and they are symmetric around zero. **This `log_return` is the target our whole project predicts.**

In [ ]:
# --- Download SPY and compute log returns ---
spy = yf.download('SPY', start=START, end=END, auto_adjust=True, progress=False)
spy = spy[['Close']].copy()
spy.columns = ['Close']
spy['log_return'] = np.log(spy['Close'] / spy['Close'].shift(1))
spy.dropna(inplace=True)

print('SPY rows:', len(spy))
print(spy.head())

### Exercise 1.1 — Visualise Price vs Returns

In [ ]:
# Exercise 1.1 — Two stacked subplots: Close price (top) and log return (bottom)
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(14, 7))

# Top: price
ax1.plot(spy.index, spy['Close'], color='steelblue', linewidth=0.8)

# Bottom: log returns
ax2.plot(spy.index, spy['log_return'], color='darkorange', linewidth=0.5, alpha=0.85)

ax1.set_title('SPY Close Price (2010–2024)')
ax2.set_title('SPY Daily Log Returns')
ax2.set_xlabel('Date')
plt.tight_layout()
plt.show()

**Which series is stationary and which is not? What does the bottom plot tell you about *when* volatility clusters (e.g. 2020)?**

The **log return** (bottom plot) is stationary — it oscillates around a near-zero mean with roughly constant variance across the full 15-year window. The **Close price** (top plot) is clearly non-stationary: it trends upward from ~\$100 in 2010 to ~\$580 in 2024, and its variance also grows with its level.

The bottom plot shows clear **volatility clustering** around the 2020 COVID crash: daily swings that were typically ±1% suddenly widen to ±5–9% for several weeks (early 2020), then gradually narrow as markets recovered. The same phenomenon, though milder, is visible around the 2011 debt-ceiling crisis and the 2018 Q4 sell-off.

### Exercise 1.2 — Short Answer

**Your answers:**

1. **Two reasons we forecast log returns instead of raw prices:**
   - *Stationarity*: raw prices trend upward (non-stationary), so a model trained on 2010–2018 data would have no statistical reference for 2024 price levels. Log returns are approximately stationary (zero mean, stable variance), making them generalisable across time.
   - *Scale invariance and additivity*: log returns are percentage moves expressed on a common scale. They are additive over time (multi-day log return = sum of daily log returns), which makes them easy to aggregate and compare across assets at any price level.

2. **Why `.shift(1)` is essential:** Computing today's return as `log(P_t / P_{t-1})` requires *yesterday's* price in the denominator; if we instead used today's price we'd trivially know the target before predicting it, introducing look-ahead bias — the `.shift(1)` ensures the denominator is always information that was already available at the start of day *t*.

---
## Section 2 — Volatility and the VIX

The **VIX** is the market's expected 30-day volatility, implied by S&P 500 option prices — often called the *fear index*. When markets panic, the VIX spikes. It is one of the most useful external signals for predicting *how big* tomorrow's move will be.

In [ ]:
# --- Download the VIX ---
vix = yf.download('^VIX', start=START, end=END, auto_adjust=True, progress=False)
vix = vix[['Close']].copy()
vix.columns = ['vix_close']
vix['vix_change_5d'] = vix['vix_close'].pct_change(5)
vix.dropna(inplace=True)
print(vix.head())

### Exercise 2.1 — Merge SPY and VIX into One Dataset

In [ ]:
# Exercise 2.1 — Merge SPY and VIX

# 1. Inner join on shared dates
df = spy.join(vix, how='inner')

# 2. Drop NaNs
df = df.dropna()

# 3. Inspect
print('Merged shape:', df.shape)
print('Columns:', list(df.columns))

# 4. Plot the VIX
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['vix_close'], color='crimson', linewidth=0.8)
ax.set_title('VIX — the market fear index (COVID spike ≈ 2020)')
ax.set_xlabel('Date')
ax.set_ylabel('VIX')
plt.tight_layout()
plt.show()

---
## Section 3 — RSI: The Relative Strength Index

**RSI** is a classic momentum oscillator bounded between 0 and 100. It compares the average size of recent *up* moves to recent *down* moves:

$$ RSI = 100 - \frac{100}{1 + RS}, \qquad RS = \frac{\text{avg gain}}{\text{avg loss}} $$

Readings above 70 suggest an asset is "overbought", below 30 "oversold".

### Exercise 3.1 — Implement `compute_rsi`

In [ ]:
# Exercise 3.1 — RSI with Wilder (EWM) smoothing
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """Relative Strength Index, computed with Wilder (EWM) smoothing."""
    # Step 1: day-to-day price difference
    delta = series.diff()
    # Step 2: separate gains (positive) and losses (made positive)
    gain = delta.clip(lower=0)
    loss = (-delta.clip(upper=0))
    # Step 3: exponentially weighted averages (Wilder smoothing: com = period-1)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    # Step 4: avoid divide-by-zero — replace zero avg_loss with NaN
    avg_loss = avg_loss.replace(0, np.nan)
    rs = avg_gain / avg_loss
    # Step 5: RSI formula
    return 100 - 100 / (1 + rs)

# Quick test: RSI should live between 0 and 100
rsi_test = compute_rsi(df['Close'])
print('RSI range:', round(rsi_test.dropna().min(), 1), 'to', round(rsi_test.dropna().max(), 1))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rsi_test.index, rsi_test, linewidth=0.8, color='purple')
ax.axhline(70, color='r', ls='--', label='Overbought (70)')
ax.axhline(30, color='g', ls='--', label='Oversold (30)')
ax.set_title('SPY 14-day RSI')
ax.set_ylim(0, 100)
ax.legend()
plt.show()

---
## Section 4 — MACD: Moving Average Convergence Divergence

MACD measures trend by subtracting a slow EMA from a fast EMA:

- `macd_line = EMA(12) - EMA(26)`
- `signal_line = EMA(9) of the macd_line`

### Exercise 4.1 — Implement `compute_macd`

In [ ]:
# Exercise 4.1 — MACD
def compute_macd(series: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
    """Returns (macd_line, signal_line)."""
    ema_fast    = series.ewm(span=fast,   adjust=False).mean()
    ema_slow    = series.ewm(span=slow,   adjust=False).mean()
    macd_line   = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line

macd_line, signal_line = compute_macd(df['Close'])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(macd_line.index, macd_line,    label='MACD line',   linewidth=0.9)
ax.plot(signal_line.index, signal_line, label='Signal line', linewidth=0.9)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('SPY MACD')
ax.legend()
plt.show()

---
## Section 5 — Bollinger Band Width

Bollinger Bands sit two standard deviations above and below a moving average. The **width** is a clean measure of recent volatility.

$$ \text{width} = \frac{(\text{mid} + 2\sigma) - (\text{mid} - 2\sigma)}{\text{mid}} = \frac{4\sigma}{\text{mid}} $$

### Exercise 5.1 — Implement `compute_bollinger`

In [ ]:
# Exercise 5.1 — Bollinger Band Width
def compute_bollinger(series: pd.Series, window: int = 20) -> pd.Series:
    """Bollinger Band width = (upper - lower) / mid."""
    mid   = series.rolling(window).mean()
    std   = series.rolling(window).std()
    upper = mid + 2 * std
    lower = mid - 2 * std
    return (upper - lower) / mid

bb = compute_bollinger(df['Close'])
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(bb.index, bb, color='teal', linewidth=0.8)
ax.set_title('SPY Bollinger Band Width (volatility proxy)')
plt.show()

---
## Section 6 — Assembling the Feature Matrix (No Leakage!)

In [ ]:
# --- Lag and rolling feature helpers (provided) ---
def compute_lag_features(series: pd.Series, lags=(1, 2, 3, 5, 10)) -> pd.DataFrame:
    return pd.DataFrame({f'lag_{lag}': series.shift(lag) for lag in lags})

def compute_rolling_features(series: pd.Series, windows=(5, 21)) -> pd.DataFrame:
    frames = {}
    for w in windows:
        frames[f'rolling_mean_{w}'] = series.shift(1).rolling(w).mean()
        frames[f'rolling_std_{w}']  = series.shift(1).rolling(w).std()
    return pd.DataFrame(frames)

print('Helpers defined. Note every column uses .shift() — that is the no-leakage guarantee.')

### Exercise 6.1 — Build the Feature Matrix

In [ ]:
# Exercise 6.1 — Build the full feature matrix
def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    close = df['Close'].shift(1)   # yesterday's close — safe to use as an input

    rsi         = compute_rsi(close).rename('rsi_14')
    macd_line_f, macd_signal = compute_macd(close)
    macd_signal = macd_signal.rename('macd_signal')
    bb_width    = compute_bollinger(close).rename('bb_width')

    lag_feats   = compute_lag_features(df['log_return'], lags=(1, 2, 3, 5, 10))
    roll_feats  = compute_rolling_features(df['log_return'], windows=(5, 21))

    vix_level   = df['vix_close'].shift(1).rename('vix_level')
    vix_change  = df['vix_change_5d'].shift(1).rename('vix_change_5d')

    # Concatenate all features + the TARGET (log_return, un-shifted)
    feature_df = pd.concat(
        [rsi, macd_signal, bb_width, lag_feats, roll_feats,
         vix_level, vix_change, df['log_return']],
        axis=1,
    )
    feature_df = feature_df.dropna()   # drop warm-up NaN rows
    return feature_df

features = build_feature_matrix(df)
print('Feature matrix shape:', features.shape)
print('Columns:', list(features.columns))
print('Total NaNs:', features.isna().sum().sum())
features.head()

### Exercise 6.2 — Prove There Is No Leakage

In [ ]:
# Exercise 6.2 — Leakage check
corr_with_target = (features.corr()['log_return']
                    .drop('log_return')
                    .reindex(features.corr()['log_return']
                             .drop('log_return').abs()
                             .sort_values(ascending=False).index))
print('Correlation of each feature with log_return (sorted by |r|):')
print(corr_with_target.round(4).to_string())
print('\nHighest absolute correlation (should be well below 1.0):')
print(corr_with_target.abs().max().round(4))

**Is the largest feature-target correlation small (e.g. < 0.2)? Why is a *low* correlation actually expected and healthy here, given that daily returns are close to random?**

Yes — the highest |correlation| is around **0.05**, well below 0.2. This is exactly what we expect and is healthy for two reasons:

1. **Daily returns are close to a random walk.** Decades of empirical finance (and the Efficient Market Hypothesis) show that each day's return has almost no predictable component from public information. A feature with |r| ≈ 0.05 is doing about as well as anyone can.

2. **Low correlation ≠ useless feature.** Even a signal with |r| = 0.04 can generate alpha when applied consistently over thousands of trades. Our multi-agent system will combine several such weak signals, and the ensemble can outperform any single indicator.

If we saw a feature with |r| > 0.9 it would be a **red flag for look-ahead bias** — that column likely contains today's return information disguised as a feature.

In [ ]:
# Save feature matrix for downstream notebooks
features.to_csv('spy_features_built.csv')
print('Saved spy_features_built.csv with', len(features), 'rows.')

---
## Bonus Challenge ⭐

In [ ]:
# BONUS A — 10-day momentum + day-of-week features
momentum_10d = df['log_return'].rolling(10).sum().shift(1).rename('momentum_10d')
dow = pd.Series(df.index.dayofweek, index=df.index, name='day_of_week')  # 0=Mon…4=Fri

features_bonus = pd.concat([features, momentum_10d, dow], axis=1).dropna()
print('Bonus A — Feature matrix with extra features:', features_bonus.shape)
print('Max |corr| with target after adding bonus features:',
      round(features_bonus.corr()['log_return'].drop('log_return').abs().max(), 4))
print('\nNo leakage introduced — both new features use .shift(1) / day-of-week is known at market open.')

In [ ]:
# BONUS B — Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(features.corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.3, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()
# Observations:
# · rolling_mean_5 and rolling_mean_21 cluster together (both momentum proxies).
# · rolling_std_5 and rolling_std_21 cluster (both volatility measures), and
#   they positively correlate with vix_level — VIX rises when realised vol rises.
# · lag features are near-orthogonal to each other (daily returns barely autocorrelate).
# · bb_width and rolling_std columns correlate positively — both capture recent volatility.